<a href="https://colab.research.google.com/github/HK25abm/Transformer-Phishing-Detection/blob/main/DeBERTa_adversarial_training_and_three_model_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import gc
import time
import torch
import numpy as np
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

In [3]:
PROJECT_DIR = "/content/drive/MyDrive/phishing_project"

PHASE2_DIR = os.path.join(
    PROJECT_DIR,
    "phase2"
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

SEED = 42

In [4]:
train_df = pd.read_csv(
    os.path.join(
        PHASE2_DIR,
        "augmented_training_dataset.csv"
    )
)

validation_df = pd.read_csv(
    os.path.join(
        PROJECT_DIR,
        "validation.csv"
    )
)

test_df = pd.read_csv(
    os.path.join(
        PROJECT_DIR,
        "test.csv"
    )
)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/phishing_project/phase2/augmented_training_dataset.csv'

In [5]:
import os

PROJECT_DIR = "/content/drive/MyDrive/phishing_project"

for root, dirs, files in os.walk(PROJECT_DIR):
    print(root)

/content/drive/MyDrive/phishing_project
/content/drive/MyDrive/phishing_project/adversarial_testing
/content/drive/MyDrive/phishing_project/saved_models
/content/drive/MyDrive/phishing_project/saved_models/bert
/content/drive/MyDrive/phishing_project/saved_models/roberta
/content/drive/MyDrive/phishing_project/saved_models/deberta
/content/drive/MyDrive/phishing_project/saved_models/deberta_fixed
/content/drive/MyDrive/phishing_project/phase1_stronger_adversarial_testing
/content/drive/MyDrive/phishing_project/phase1_contextual_adversarial
/content/drive/MyDrive/phishing_project/results
/content/drive/MyDrive/phishing_project/results/deberta_fixed
/content/drive/MyDrive/phishing_project/final_phase1_results
/content/drive/MyDrive/phishing_project/phase2_adversarial_training
/content/drive/MyDrive/phishing_project/phase2_adversarial_training/saved_models
/content/drive/MyDrive/phishing_project/phase2_adversarial_training/saved_models/bert_adversarially_trained
/content/drive/MyDrive/phi

In [6]:
import os

for root, dirs, files in os.walk(PROJECT_DIR):
    for file in files:
        if "augmented" in file.lower():
            print(os.path.join(root, file))

/content/drive/MyDrive/phishing_project/phase2_adversarial_training/augmented_training_dataset.csv


In [7]:
PHASE2_DIR = "/content/drive/MyDrive/phishing_project/phase2_adversarial_training"

train_df = pd.read_csv(
    os.path.join(
        PHASE2_DIR,
        "augmented_training_dataset.csv"
    )
)

print(train_df.shape)
print(train_df["label"].value_counts())

(15354, 6)
label
1    8354
0    7000
Name: count, dtype: int64


In [8]:
PROJECT_DIR = "/content/drive/MyDrive/phishing_project"

PHASE2_DIR = os.path.join(
    PROJECT_DIR,
    "phase2_adversarial_training"
)

In [9]:
validation_df = pd.read_csv(
    os.path.join(
        PROJECT_DIR,
        "validation.csv"
    )
)

test_df = pd.read_csv(
    os.path.join(
        PROJECT_DIR,
        "test.csv"
    )
)

In [10]:
import os
import gc
import time
import torch
import numpy as np
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

PROJECT_DIR = "/content/drive/MyDrive/phishing_project"

PHASE2_DIR = os.path.join(
    PROJECT_DIR,
    "phase2_adversarial_training"
)

DEBERTA_FIXED_PATH = os.path.join(
    PROJECT_DIR,
    "saved_models",
    "deberta_fixed"
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

SEED = 42

print("DEVICE:", DEVICE)
print("PHASE2_DIR:", PHASE2_DIR)
print("DEBERTA_FIXED_PATH:", DEBERTA_FIXED_PATH)

DEVICE: cuda
PHASE2_DIR: /content/drive/MyDrive/phishing_project/phase2_adversarial_training
DEBERTA_FIXED_PATH: /content/drive/MyDrive/phishing_project/saved_models/deberta_fixed


In [11]:
augmented_train_v2 = pd.read_csv(
    os.path.join(
        PHASE2_DIR,
        "augmented_training_dataset.csv"
    )
)

validation_df = pd.read_csv(
    os.path.join(
        PROJECT_DIR,
        "validation.csv"
    )
)

test_df = pd.read_csv(
    os.path.join(
        PROJECT_DIR,
        "test.csv"
    )
)

print("Train:", augmented_train_v2.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

print("\nTraining labels:")
print(augmented_train_v2["label"].value_counts())

Train: (15354, 6)
Validation: (3000, 6)
Test: (3000, 6)

Training labels:
label
1    8354
0    7000
Name: count, dtype: int64


In [12]:
class EmailDataset(torch.utils.data.Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_length=256
    ):
        self.texts = (
            pd.Series(texts)
            .fillna("")
            .astype(str)
            .tolist()
        )

        self.labels = (
            pd.Series(labels)
            .astype(int)
            .tolist()
        )

        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length
        )

        encoding["labels"] = self.labels[idx]

        return encoding

In [13]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

def compute_metrics_adv_training(eval_pred):

    logits, labels = eval_pred

    logits = np.asarray(
        logits,
        dtype=np.float64
    )

    logits = logits - np.max(
        logits,
        axis=1,
        keepdims=True
    )

    exp_logits = np.exp(logits)

    probs = exp_logits / np.sum(
        exp_logits,
        axis=1,
        keepdims=True
    )

    predictions = np.argmax(
        probs,
        axis=1
    )

    return {
        "accuracy": accuracy_score(
            labels,
            predictions
        ),

        "precision": precision_score(
            labels,
            predictions,
            zero_division=0
        ),

        "recall": recall_score(
            labels,
            predictions,
            zero_division=0
        ),

        "f1": f1_score(
            labels,
            predictions,
            zero_division=0
        ),

        "roc_auc": roc_auc_score(
            labels,
            probs[:, 1]
        )
    }

In [14]:
deberta_tokenizer = AutoTokenizer.from_pretrained(
    DEBERTA_FIXED_PATH
)

deberta_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        DEBERTA_FIXED_PATH,
        torch_dtype=torch.float32
    )
)

deberta_model.to(DEVICE)

print("Fixed DeBERTa loaded.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/202 [00:01<?, ?it/s]

Fixed DeBERTa loaded.


In [15]:
deberta_train_dataset = EmailDataset(
    augmented_train_v2["text"],
    augmented_train_v2["label"],
    deberta_tokenizer,
    max_length=256
)

deberta_validation_dataset = EmailDataset(
    validation_df["text"],
    validation_df["label"],
    deberta_tokenizer,
    max_length=256
)

deberta_test_dataset = EmailDataset(
    test_df["text"],
    test_df["label"],
    deberta_tokenizer,
    max_length=256
)

deberta_collator = DataCollatorWithPadding(
    tokenizer=deberta_tokenizer
)

print("Train:", len(deberta_train_dataset))
print("Validation:", len(deberta_validation_dataset))
print("Test:", len(deberta_test_dataset))

Train: 15354
Validation: 3000
Test: 3000


In [16]:
DEBERTA_CHECKPOINT_DIR = (
    "/content/deberta_adv_training_checkpoints"
)

DEBERTA_DEFENDED_PATH = os.path.join(
    PROJECT_DIR,
    "final_phase2_results",
    "deberta_final_defended"
)

os.makedirs(
    DEBERTA_DEFENDED_PATH,
    exist_ok=True
)

print(DEBERTA_DEFENDED_PATH)

/content/drive/MyDrive/phishing_project/final_phase2_results/deberta_final_defended


In [17]:
deberta_args = TrainingArguments(

    output_dir=
        DEBERTA_CHECKPOINT_DIR,

    learning_rate=
        5e-6,

    per_device_train_batch_size=
        4,

    gradient_accumulation_steps=
        4,

    per_device_eval_batch_size=
        8,

    num_train_epochs=
        1,

    weight_decay=
        0.01,

    warmup_ratio=
        0.05,

    max_grad_norm=
        1.0,

    eval_strategy=
        "epoch",

    save_strategy=
        "epoch",

    logging_strategy=
        "epoch",

    load_best_model_at_end=
        True,

    metric_for_best_model=
        "f1",

    greater_is_better=
        True,

    save_total_limit=
        1,

    report_to=
        "none",

    seed=
        42,

    data_seed=
        42,

    fp16=
        False,

    bf16=
        False
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [18]:
deberta_trainer = Trainer(

    model=
        deberta_model,

    args=
        deberta_args,

    train_dataset=
        deberta_train_dataset,

    eval_dataset=
        deberta_validation_dataset,

    processing_class=
        deberta_tokenizer,

    data_collator=
        deberta_collator,

    compute_metrics=
        compute_metrics_adv_training
)

In [19]:
start_time = time.time()

deberta_trainer.train()

deberta_adv_training_time = (
    time.time()
    -
    start_time
)

print(
    "DeBERTa adversarial training time:",
    round(
        deberta_adv_training_time,
        2
    ),
    "seconds"
)

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.079691,0.061749,0.992000,0.994638,0.989333,0.991979,0.999565


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DeBERTa adversarial training time: 1014.45 seconds


In [20]:
deberta_trainer.model.save_pretrained(
    DEBERTA_DEFENDED_PATH
)

deberta_tokenizer.save_pretrained(
    DEBERTA_DEFENDED_PATH
)

print("DeBERTa model saved.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

DeBERTa model saved.


In [21]:
import os

print(os.listdir(DEBERTA_DEFENDED_PATH))

['config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json']


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import torch
import numpy as np
import pandas as pd

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

In [3]:
PROJECT_DIR = "/content/drive/MyDrive/phishing_project"

DEBERTA_DEFENDED_PATH = os.path.join(
    PROJECT_DIR,
    "final_phase2_results",
    "deberta_final_defended"
)

PHASE1_ADV_DIR = os.path.join(
    PROJECT_DIR,
    "phase1_contextual_adversarial"
)

FINAL_PHASE2_DIR = os.path.join(
    PROJECT_DIR,
    "final_phase2_results"
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("DEVICE:", DEVICE)
print("DeBERTa path:", DEBERTA_DEFENDED_PATH)

DEVICE: cuda
DeBERTa path: /content/drive/MyDrive/phishing_project/final_phase2_results/deberta_final_defended


In [4]:
test_df = pd.read_csv(
    os.path.join(
        PROJECT_DIR,
        "test.csv"
    )
)

test_df["text"] = (
    test_df["text"]
    .fillna("")
    .astype(str)
)

test_df["label"] = (
    test_df["label"]
    .astype(int)
)

print(test_df.shape)
print(test_df["label"].value_counts())

(3000, 6)
label
0    1500
1    1500
Name: count, dtype: int64


In [5]:
deberta_def_tokenizer = AutoTokenizer.from_pretrained(
    DEBERTA_DEFENDED_PATH
)

deberta_def_model = AutoModelForSequenceClassification.from_pretrained(
    DEBERTA_DEFENDED_PATH,
    torch_dtype=torch.float32
)

deberta_def_model.to(DEVICE)
deberta_def_model.eval()

print("Defended DeBERTa loaded.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Defended DeBERTa loaded.


In [6]:
def get_probabilities_batch(
    model,
    tokenizer,
    texts,
    batch_size=8,
    max_length=256
):

    model.eval()
    all_probs = []

    for start in range(
        0,
        len(texts),
        batch_size
    ):

        batch_texts = texts[
            start:start + batch_size
        ]

        encoding = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )

        encoding = {
            k: v.to(DEVICE)
            for k, v in encoding.items()
        }

        with torch.no_grad():

            logits = model(
                **encoding
            ).logits

            probs = torch.softmax(
                logits.float(),
                dim=1
            )

        all_probs.append(
            probs.cpu().numpy()
        )

    return np.concatenate(
        all_probs,
        axis=0
    )

In [7]:
def evaluate_dataset_model(
    model,
    tokenizer,
    dataframe,
    batch_size=8
):

    probs = get_probabilities_batch(
        model,
        tokenizer,
        dataframe["text"].tolist(),
        batch_size=batch_size
    )

    y_true = (
        dataframe["label"]
        .astype(int)
        .values
    )

    y_pred = np.argmax(
        probs,
        axis=1
    )

    metrics = {
        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "precision":
            precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "recall":
            recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "f1":
            f1_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        "roc_auc":
            roc_auc_score(
                y_true,
                probs[:, 1]
            )
    }

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    return metrics, cm, probs, y_pred

In [8]:
deberta_clean_metrics, \
deberta_clean_cm, \
deberta_clean_probs, \
deberta_clean_pred = evaluate_dataset_model(
    deberta_def_model,
    deberta_def_tokenizer,
    test_df,
    batch_size=8
)

print("DEFENDED DEBERTA — CLEAN")

print(
    deberta_clean_metrics
)

print(
    "\nConfusion Matrix:"
)

print(
    deberta_clean_cm
)

DEFENDED DEBERTA — CLEAN
{'accuracy': 0.9896666666666667, 'precision': 0.991304347826087, 'recall': 0.988, 'f1': 0.9896494156928214, 'roc_auc': np.float64(0.9987506666666667)}

Confusion Matrix:
[[1487   13]
 [  18 1482]]


In [9]:
DEBERTA_ADV_PATH = os.path.join(
    PHASE1_ADV_DIR,
    "deberta_contextual_adversarial_results.csv"
)

deberta_phase1_adv_df = pd.read_csv(
    DEBERTA_ADV_PATH
)

print(
    deberta_phase1_adv_df.shape
)

print(
    deberta_phase1_adv_df.columns.tolist()
)

(297, 10)
['original_text', 'adversarial_text', 'original_probability', 'adversarial_probability', 'confidence_drop', 'semantic_similarity', 'num_changes', 'changes', 'successful_evasion', 'sample_id']


In [10]:
def build_model_adversarial_test(
    original_test_df,
    attack_results_df
):

    adv_test = original_test_df.copy()

    attack_map = dict(
        zip(
            attack_results_df["sample_id"],
            attack_results_df["adversarial_text"]
        )
    )

    adv_test["text"] = adv_test.apply(
        lambda row: (
            attack_map.get(
                row["sample_id"],
                row["text"]
            )
            if row["label"] == 1
            else row["text"]
        ),
        axis=1
    )

    return adv_test

In [11]:
deberta_frozen_adv_test_df = build_model_adversarial_test(
    test_df,
    deberta_phase1_adv_df
)

print(
    deberta_frozen_adv_test_df[
        "label"
    ].value_counts()
)

label
0    1500
1    1500
Name: count, dtype: int64


In [12]:
deberta_adv_metrics, \
deberta_adv_cm, \
deberta_adv_probs, \
deberta_adv_pred = evaluate_dataset_model(
    deberta_def_model,
    deberta_def_tokenizer,
    deberta_frozen_adv_test_df,
    batch_size=8
)

print(
    "DEFENDED DEBERTA — ADVERSARIAL"
)

print(
    deberta_adv_metrics
)

print(
    "\nConfusion Matrix:"
)

print(
    deberta_adv_cm
)

DEFENDED DEBERTA — ADVERSARIAL
{'accuracy': 0.9876666666666667, 'precision': 0.9912693082605776, 'recall': 0.984, 'f1': 0.9876212780194045, 'roc_auc': np.float64(0.998568)}

Confusion Matrix:
[[1487   13]
 [  24 1476]]


In [13]:
deberta_phase2_comparison = pd.DataFrame([
    {
        "Condition": "Baseline Clean",
        "Accuracy": 0.9880,
        "Recall": 0.9853,
        "F1": 0.9880,
        "ROC_AUC": 0.9986
    },
    {
        "Condition": "Baseline Adversarial",
        "Accuracy": 0.9847,
        "Recall": 0.9787,
        "F1": 0.9846,
        "ROC_AUC": 0.9984
    },
    {
        "Condition": "Defended Clean",
        "Accuracy": deberta_clean_metrics["accuracy"],
        "Recall": deberta_clean_metrics["recall"],
        "F1": deberta_clean_metrics["f1"],
        "ROC_AUC": deberta_clean_metrics["roc_auc"]
    },
    {
        "Condition": "Defended Adversarial",
        "Accuracy": deberta_adv_metrics["accuracy"],
        "Recall": deberta_adv_metrics["recall"],
        "F1": deberta_adv_metrics["f1"],
        "ROC_AUC": deberta_adv_metrics["roc_auc"]
    }
])

display(
    deberta_phase2_comparison.round(4)
)

,Condition,Accuracy,Recall,F1,ROC_AUC
0,Baseline Clean,0.9880,0.9853,0.9880,0.9986
1,Baseline Adversarial,0.9847,0.9787,0.9846,0.9984
2,Defended Clean,0.9897,0.9880,0.9896,0.9988
3,Defended Adversarial,0.9877,0.9840,0.9876,0.9986


In [14]:
deberta_robustness_gain = (
    deberta_adv_metrics["recall"]
    - 0.9787
)

print(
    "DeBERTa robustness gain:",
    round(
        deberta_robustness_gain,
        4
    )
)

DeBERTa robustness gain: 0.0053


In [15]:
FINAL_DEBERTA_DIR = os.path.join(
    PROJECT_DIR,
    "final_phase2_results",
    "deberta_final_defended"
)

os.makedirs(
    FINAL_DEBERTA_DIR,
    exist_ok=True
)

In [16]:
pd.DataFrame(
    [deberta_clean_metrics]
).to_csv(
    os.path.join(
        FINAL_DEBERTA_DIR,
        "clean_metrics.csv"
    ),
    index=False
)

pd.DataFrame(
    [deberta_adv_metrics]
).to_csv(
    os.path.join(
        FINAL_DEBERTA_DIR,
        "adversarial_metrics.csv"
    ),
    index=False
)

deberta_phase2_comparison.to_csv(
    os.path.join(
        FINAL_DEBERTA_DIR,
        "robustness_comparison.csv"
    ),
    index=False
)

In [17]:
deberta_summary = pd.DataFrame([
    {
        "model": "DeBERTa",
        "baseline_clean_accuracy": 0.9880,
        "defended_clean_accuracy":
            deberta_clean_metrics["accuracy"],

        "baseline_adv_accuracy": 0.9847,
        "defended_adv_accuracy":
            deberta_adv_metrics["accuracy"],

        "baseline_adv_recall": 0.9787,
        "defended_adv_recall":
            deberta_adv_metrics["recall"],

        "robustness_gain":
            deberta_adv_metrics["recall"]
            - 0.9787
    }
])

deberta_summary.to_csv(
    os.path.join(
        FINAL_DEBERTA_DIR,
        "experiment_summary.csv"
    ),
    index=False
)

display(
    deberta_summary.round(4)
)

,model,baseline_clean_accuracy,defended_clean_accuracy,baseline_adv_accuracy,defended_adv_accuracy,baseline_adv_recall,defended_adv_recall,robustness_gain
0,DeBERTa,0.988,0.9897,0.9847,0.9877,0.9787,0.984,0.0053


In [18]:
deberta_clean_predictions = test_df.copy()

deberta_clean_predictions[
    "prediction"
] = deberta_clean_pred

deberta_clean_predictions[
    "prob_legitimate"
] = deberta_clean_probs[:, 0]

deberta_clean_predictions[
    "prob_phishing"
] = deberta_clean_probs[:, 1]

deberta_clean_predictions.to_csv(
    os.path.join(
        FINAL_DEBERTA_DIR,
        "clean_predictions.csv"
    ),
    index=False
)

In [19]:
deberta_adv_predictions = (
    deberta_frozen_adv_test_df.copy()
)

deberta_adv_predictions[
    "prediction"
] = deberta_adv_pred

deberta_adv_predictions[
    "prob_legitimate"
] = deberta_adv_probs[:, 0]

deberta_adv_predictions[
    "prob_phishing"
] = deberta_adv_probs[:, 1]

deberta_adv_predictions.to_csv(
    os.path.join(
        FINAL_DEBERTA_DIR,
        "adversarial_predictions.csv"
    ),
    index=False
)

In [20]:
np.savetxt(
    os.path.join(
        FINAL_DEBERTA_DIR,
        "clean_confusion_matrix.csv"
    ),
    deberta_clean_cm,
    fmt="%d",
    delimiter=","
)

np.savetxt(
    os.path.join(
        FINAL_DEBERTA_DIR,
        "adversarial_confusion_matrix.csv"
    ),
    deberta_adv_cm,
    fmt="%d",
    delimiter=","
)

In [21]:
final_phase2_comparison = pd.DataFrame([
    {
        "model": "BERT",

        "baseline_clean_accuracy": 0.9873,
        "defended_clean_accuracy": 0.9870,

        "baseline_adv_accuracy": 0.9823,
        "defended_adv_accuracy": 0.9833,

        "baseline_adv_recall": 0.9773,
        "defended_adv_recall": 0.9820,

        "baseline_adv_f1": 0.9822,
        "defended_adv_f1": 0.9833,

        "robustness_gain": 0.0047
    },

    {
        "model": "RoBERTa",

        "baseline_clean_accuracy": 0.9890,
        "defended_clean_accuracy": 0.9907,

        "baseline_adv_accuracy": 0.9853,
        "defended_adv_accuracy": 0.9887,

        "baseline_adv_recall": 0.9773,
        "defended_adv_recall": 0.9827,

        "baseline_adv_f1": 0.9852,
        "defended_adv_f1": 0.9886,

        "robustness_gain": 0.0054
    },

    {
        "model": "DeBERTa",

        "baseline_clean_accuracy": 0.9880,
        "defended_clean_accuracy": 0.9897,

        "baseline_adv_accuracy": 0.9847,
        "defended_adv_accuracy": 0.9877,

        "baseline_adv_recall": 0.9787,
        "defended_adv_recall": 0.9840,

        "baseline_adv_f1": 0.9846,
        "defended_adv_f1": 0.9876,

        "robustness_gain": 0.0053
    }
])

display(
    final_phase2_comparison.round(4)
)

,model,baseline_clean_accuracy,defended_clean_accuracy,baseline_adv_accuracy,defended_adv_accuracy,baseline_adv_recall,defended_adv_recall,baseline_adv_f1,defended_adv_f1,robustness_gain
0,BERT,0.9873,0.9870,0.9823,0.9833,0.9773,0.9820,0.9822,0.9833,0.0047
1,RoBERTa,0.9890,0.9907,0.9853,0.9887,0.9773,0.9827,0.9852,0.9886,0.0054
2,DeBERTa,0.9880,0.9897,0.9847,0.9877,0.9787,0.9840,0.9846,0.9876,0.0053


In [22]:
FINAL_PHASE2_RESULTS_DIR = os.path.join(
    PROJECT_DIR,
    "final_phase2_results"
)

final_phase2_comparison.to_csv(
    os.path.join(
        FINAL_PHASE2_RESULTS_DIR,
        "final_adversarial_training_comparison.csv"
    ),
    index=False
)

print(
    "Final Phase 2 comparison saved."
)

Final Phase 2 comparison saved.


In [23]:
import os

print(os.listdir(FINAL_PHASE2_RESULTS_DIR))

['bert_final_defended', 'roberta_final_defended', 'deberta_final_defended', 'final_adversarial_training_comparison.csv']


In [24]:
saved_df = pd.read_csv(
    os.path.join(
        FINAL_PHASE2_RESULTS_DIR,
        "final_adversarial_training_comparison.csv"
    )
)

display(saved_df.round(4))

,model,baseline_clean_accuracy,defended_clean_accuracy,baseline_adv_accuracy,defended_adv_accuracy,baseline_adv_recall,defended_adv_recall,baseline_adv_f1,defended_adv_f1,robustness_gain
0,BERT,0.9873,0.9870,0.9823,0.9833,0.9773,0.9820,0.9822,0.9833,0.0047
1,RoBERTa,0.9890,0.9907,0.9853,0.9887,0.9773,0.9827,0.9852,0.9886,0.0054
2,DeBERTa,0.9880,0.9897,0.9847,0.9877,0.9787,0.9840,0.9846,0.9876,0.0053
